# Chapter 11 — Reinforcement learning: practice instead of imitation

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 11 — Reinforcement learning: practice instead of imitation

**Sources:** Karpathy's "Deep Dive into LLMs like ChatGPT" `[transcript]` and *The Little Book of Reinforcement Learning* `[little-book]`. **Runs on:** the GRPO experiment takes 30 seconds on a GPU and a few minutes on a CPU.

### The problem

Chapter 10 ended on two walls. SFT can only reproduce behaviour someone demonstrated, and demonstrations cost human time. For a maths problem nobody has solved, or a coding style nobody has written down, there is nothing to imitate.

Reinforcement learning removes the demonstration requirement. You supply a way of *scoring* an attempt, and the model searches for behaviour that scores well. The little book puts the trade precisely: SFT "requires demonstrations. RL on the other hand needs no demonstrations, only a reward function that scores model outputs and lets the model search for behaviour that scores well" `[little-book, ch5]`.

That difference has a consequence worth stating plainly: because the signal comes from a score rather than a fixed human-written target, **RL can in principle exceed any individual human demonstrator** `[little-book, ch5]`.

> **Say it to a six-year-old.** There are two ways to get good at something. One is copying: you watch someone tie their shoes and you do what they did. That only works if someone shows you, and you can never get better at it than them. The other way is practising: you try, you see if it worked, and if it did you do more of that. Nobody has to show you. You can end up better than anyone who could have taught you. Computers learn both ways, and this chapter is the practising one.

### 11.1 Reinforcement learning from zero

Everything so far in this book has been **supervised learning**: for each input there is a correct answer, and the loss measures the distance to it. RL has no correct answers, only outcomes that turn out better or worse.

The pieces, each defined before it is used `[little-book, ch1]`:

- **Agent** — the thing making decisions. Here, the language model.
- **Environment** — everything the agent interacts with, which responds to what the agent does.
- **State** (written *S*) — the situation the agent is currently in.
- **Action** (written *A*) — what the agent does next, chosen from the actions available.
- **Reward** (written *R*) — a number scoring what happened. Higher is better. This is the *only* feedback.
- **Policy** (written *π*, "pi") — the agent's strategy: a rule mapping a state to a probability distribution over actions. Training means improving π.
- **Trajectory** or **rollout** — one complete run from start to finish: state, action, state, action, and eventually a reward.
- **Return** — the total reward collected over a trajectory. The thing being maximized.

**The interaction loop**: the agent observes a state, picks an action, the environment moves to a new state and possibly hands back a reward, and this repeats until the episode ends.

**Analogy.** Learning to cook without a recipe. The kitchen is the environment, what is currently in the pan is the state, adding salt is an action, and the reward comes at the end when someone tastes it. You do not get told "you should have added the salt 40 seconds earlier." You get one number at the end, and you have to work out which of your fifty decisions deserves the credit.

That last sentence is the central difficulty of RL, and it has a name: the **credit assignment problem**. A sparse reward at the end of a long trajectory has to be attributed to the individual actions that earned it.

### 11.2 A language model as an RL problem

The translation is exact, and once you see it the rest of the chapter follows `[little-book, §5.1]`:

| RL concept | In an LLM |
|---|---|
| **Policy** π<sub>θ</sub> | The language model itself. It already outputs a probability distribution over next tokens, which *is* a policy |
| **State** S<sub>t</sub> | The prompt plus every token generated so far |
| **Action** A<sub>t</sub> | The next token, chosen from the vocabulary |
| **Transition** | Deterministic: append the token to the state. Nothing random happens |
| **Reward** | Sparse and terminal: a verifier scores the finished response and returns one number |
| **Episode** | One complete response, ending at the end-of-sequence token or a length cap |

**The policy was already there.** This is the part worth appreciating. You do not have to build anything new to do RL on a language model: a softmax over the vocabulary is a policy over actions, so the model you trained in Chapter 9 is already an RL agent that has never been given a reward.

Two features make this an unusually convenient RL problem. Transitions are deterministic, so all the randomness is in the policy's own sampling. And the environment is trivial: appending a token cannot fail.

One feature makes it unusually hard. The action space is the whole vocabulary, around 50,000 to 100,000 actions at every single step, and a reward arrives only after hundreds of them.

### 11.3 Two generations of RL on language models

The field has done this twice, for different reasons `[little-book, ch5]`.

**First generation: alignment (RLHF), around 2022.** Turn a base model into an assistant. Train a **reward model** to predict which of two responses a human would prefer, then use RL to push the policy toward responses that reward model scores highly. This is **RLHF**, Reinforcement Learning from Human Feedback, and it is what made ChatGPT feel like ChatGPT.

Its limit is the signal. Predicting human preference is "quite a shallow signal determined mostly by tone and style," and it "gives the model no incentive to reason, plan, or develop any new capability" `[little-book, ch5]`. You get something pleasant to talk to, not something that can think.

**Second generation: reasoning (RLVR), from 2024.** Use rewards that can be checked mechanically: does the maths answer match, do the tests pass. This is **RLVR**, Reinforcement Learning with Verifiable Rewards. Karpathy's framing is that thinking "emerges in the process of the optimization when we basically run RL on many math and code problems that have verifiable solutions" `[transcript]`.

The word *emerges* is doing real work there. Nobody demonstrated step-by-step reasoning and nobody rewarded it directly. The reward is only on the final answer. Reasoning appears because it is instrumentally useful for getting the final answer right.

Two properties make RLVR practical at scale where RLHF was not: the reward is harder to game, and it is more "interesting" to optimize against, in that pushing on it tends to develop real capability. As a result RLVR is run at a non-trivial fraction of pretraining compute, above 10% `[little-book, ch5]`.

**The dividing line is verifiability**, and it runs through the rest of this chapter:

| | Verifiable | Unverifiable |
|---|---|---|
| Examples | maths, code, formal logic | creative writing, advice, tone |
| Reward source | a function you can write | a model trained on human comparisons |
| Can it be gamed? | Hard | Easily |
| Method | GRPO and relatives | RLHF, DPO |

### 11.4 How a policy improves: the idea behind every method here

The whole family of methods rests on one sentence: **make the actions that led to good outcomes more likely, and the ones that led to bad outcomes less likely.**

Four refinements turn that sentence into GRPO, and each fixes a specific problem with the one before `[little-book, ch4]`.

**1. The basic policy gradient (REINFORCE).** Sample a trajectory, get its return, and adjust the parameters to increase the log-probability of every action taken, scaled by that return. Good rollout, all its tokens become more likely.

The problem: **variance**. If every rollout scores between 8 and 10, every action gets pushed up, just by different amounts. The learning signal is buried in a large constant.

**2. Subtract a baseline.** Instead of scaling by the raw return, scale by the return *minus a reference value*. Now a rollout scoring 8 when the average is 9 gets pushed **down**, which is the correct instruction. This quantity, "how much better than expected," is the **advantage**.

Subtracting a baseline does not bias the result, provided the baseline does not depend on the action taken. It only reduces variance. That is the single most important trick in policy-gradient methods.

**3. Do not step too far (PPO).** Policy-gradient estimates are only valid near the policy that produced the data. Take a large step and your data no longer describes the policy you now have. **PPO** (Proximal Policy Optimization) handles this by clipping: it computes the ratio between the new policy's probability for an action and the old one's, and refuses to let a single update move that ratio outside roughly 0.8 to 1.2. If an update wants to make a token twenty times more likely, it gets 1.2 times more likely, and that is the end of it.

**4. Drop the critic (GRPO).** PPO estimates the advantage with a learned **value function**, a second network that predicts expected return from a state. For LLMs that critic is "typically as large as the policy," expensive in memory, and hard to train because the reward is sparse and arrives only at the end of long trajectories `[little-book, §5.2]`.

**GRPO** (Group Relative Policy Optimization) removes it with an idea that is obvious in hindsight: to know whether a response is better than average, generate several responses to the same prompt and compare them to each other. The baseline is the group's mean reward. No second network.

For a group of G responses with rewards R₁…R<sub>G</sub>, the advantage of response *i* is:

**Expected output:**

```
advantage_i = R_i − mean(R)
```

optionally divided by the group's standard deviation. That is the whole of it `[little-book, §5.2]`.

**One more component: the leash.** Left alone, a policy chasing a reward will drift into degenerate text that scores well and reads like nothing. So the objective includes a **KL penalty** against a frozen **reference policy**, usually the SFT model you started from. KL divergence measures how far one distribution has moved from another; penalizing it keeps the policy recognizably close to where it started. This "prevents collapse to degenerate token distributions that exploit the reward without producing readable text" `[little-book, §5.1]`. You will watch that happen in section 11.6.

**GRPO also generalizes beyond language.** It applies to any environment where only terminal rewards are available and you can start multiple rollouts from the same state `[little-book, §5.2]`.

### 11.5 GRPO, implemented and run

Small enough to read, real enough to work. The task: make the names model from Chapter 2's dataset produce names starting with `k` and ending with `a`. The verifier is four lines of Python, and **there are no demonstrations anywhere in this process**.

**Run it.**

In [ ]:
def reward(name: str) -> float:
    """A programmatic grader. No reward model, no human labels, no gradients.
    Stands in for 'check the maths answer' in a real RLVR setup."""
    if len(name) < 3:
        return 0.0
    return 1.0 * (name[0] == 'k' and name[-1] == 'a')

ref = copy.deepcopy(model).eval()          # frozen reference for the KL leash
for p in ref.parameters():
    p.requires_grad = False

G, EPS, BETA, LR = 64, 0.2, 0.02, 1e-5
opt = torch.optim.AdamW(model.parameters(), lr=LR)

def logprobs_of(m, idx):
    """log pi(token_t | everything before t), for each generated token."""
    lp = F.log_softmax(m(idx[:, :-1]), -1)
    return lp.gather(-1, idx[:, 1:].unsqueeze(-1)).squeeze(-1)

for step in range(600):
    # 1. roll out G responses from the same starting state
    with torch.no_grad():
        idx, texts = model.sample(G)
        old_lp = logprobs_of(model, idx)
    R = torch.tensor([reward(t) for t in texts], device=device)

    # 2. group-relative advantage: the baseline is the group's own mean
    A = R - R.mean()
    if R.std() > 0:
        A = A / (R.std() + 1e-8)

    # 3. clipped update, plus a KL leash to the frozen reference
    new_lp = logprobs_of(model, idx)
    ratio = (new_lp - old_lp).exp()
    pg = -torch.min(ratio * A.unsqueeze(1),
                    ratio.clamp(1 - EPS, 1 + EPS) * A.unsqueeze(1))
    with torch.no_grad():
        ref_lp = logprobs_of(ref, idx)
    loss = ((pg + BETA * (new_lp - ref_lp)) * live).sum() / live.sum()

    opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

**What you should see:**

**Expected output:**

```
base model samples: ['lioni', 'hawre', 'maushawn', 'jayonna', 'makai', 'abarami']
base model satisfies the verifier 0.6% of the time
  grpo    0  reward 0.000  avg25 0.000  kl +0.0000  [0s]
  grpo  100  reward 0.141  avg25 0.059  kl +0.0205  [3s]
  grpo  200  reward 0.406  avg25 0.309  kl +0.1125  [4s]
  grpo  300  reward 0.812  avg25 0.701  kl +0.2518  [6s]
  grpo  400  reward 0.891  avg25 0.900  kl +0.3310  [8s]
  grpo  599  reward 0.969  avg25 0.976  kl +0.5259  [11s]

before GRPO: 0.6%   after GRPO: 96.1%
```

[verified, 11 seconds on one GPU]

**From 0.6% to 96.1%, from a four-line scoring function and no examples of the target behaviour whatsoever.**

Trace what happened in the first hundred steps, because it is the mechanism in miniature. At step 0 all 64 rollouts scored zero, so the advantage was zero for every one of them and nothing was learned. There was no gradient because there was no *difference*. Then a rollout happened to start with `k` and end with `a`, scoring 1 against a group mean near 0.02, giving it a large positive advantage, and every token in it became more likely. That is the entire algorithm: **rare accidental successes get amplified until they stop being accidental.**

This is also why the reward must be achievable by chance at the start. A verifier the base model satisfies 0% of the time produces zero gradient forever. In real RLVR pipelines this is exactly why prompts are filtered so that some rollouts succeed and some fail; batches where every rollout scores identically carry no signal `[little-book, §5.3]`.

### 11.6 Reward hacking, watched live

Run the same experiment and look at what it produces at the end:

**Expected output:**

```
samples after GRPO: ['kaiaeaaa', 'kaina', 'keauana', 'kariaa', 'kayaaa',
                     'nalaha', 'kakaara', 'kayaana']
```

[verified]

`kaiaeaaa`. `kayaaa`. `kariaa`. These satisfy the verifier perfectly and have stopped being names. The model found that the cheapest route to reward is `k`, then a pile of vowels, then `a`, and the verifier has no opinion about whether the result is pronounceable.

**This is reward hacking**, also called Goodhart's law: the model optimizes the measure, not the thing the measure was standing in for. It is the central practical risk of RL, and here it took eleven seconds to appear.

**Measuring the drift.** "It stopped looking like a name" can be quantified: score the policy's samples under the *frozen base model* and see how surprised it is. Low is name-like, high is drift.

**What you should see**, comparing KL penalty strengths at 600 steps:

| β (KL penalty) | Reward | Naturalness (NLL under base model) |
|---|---|---|
| 0.0 (no leash) | 1.8% → **95.7%** | 1.942 → 2.218 |
| 2.0 (strong leash) | 1.8% → **62.1%** | 1.942 → **2.000** |

[verified]

There is the trade-off, measured. Without a leash the policy gets 96% of the reward and drifts furthest from natural text. With a strong leash it stays close to the base model's notion of a name and gives up a third of the reward.

**An honest note.** I first ran this comparison at β = 0.02 and β = 0.2 and got results indistinguishable from β = 0. Printing the loss components explained why: at β = 0.2 the KL term was worth about 0.01 against a policy-gradient term of about 0.03, too small to matter, while at β = 2.0 the KL term ends up roughly **59× larger** than the policy-gradient term and dominates completely. β is not a dial with a natural scale; it has to be tuned against the size of your actual gradient signal. [verified]

**Whether degeneration happens at all is seed-dependent.** On another run with the same settings the model reached 97% while still producing `kamala`, `kamara`, `koda`. Reward hacking is a tendency, not a certainty, which is precisely what makes it dangerous to check for by eye.

### 11.7 SFT against RL, on the identical task

Chapter 10 and this chapter targeted the same behaviour with the same base model, which makes them directly comparable.

| Method | Demonstrations needed | Target rate | Distinct names per 512 samples |
|---|---|---|---|
| Base model | — | 0.6–1.4% | 509 |
| SFT, 493 demonstrations | 493 | 96.5% | 378 |
| SFT, 20 demonstrations | 20 | 98.2% | **53** |
| **GRPO, no demonstrations** | **0** | **96.1%** | **376** |

[verified]

**GRPO matched SFT-with-493-demonstrations on both axes while using none.** It only ever saw a function that returns 0 or 1.

That is the case for RL in one table. And the row above it is the case against: SFT on 20 demonstrations scored *highest* on the target while collapsing to 53 distinct outputs. Any single metric can be satisfied by a model that has quietly destroyed everything you were not measuring.

### 11.8 When there is nothing to verify

Most of what people want from an assistant cannot be checked by a function. Is this email polite? Is this explanation clear? There is no grader to write.

The answer is to learn the grader from people. Show a human two responses, ask which is better, and fit a **reward model** to those comparisons. Then optimize against it.

**DPO** (Direct Preference Optimization) simplifies this by removing the separate reward model. It optimizes the policy directly on preference pairs, using a loss that raises the policy's log-probability of the preferred response relative to the rejected one, both measured against the frozen reference. The quantity it maximizes is the **implicit reward margin**:

**Expected output:**

```
margin = [log π(winner) − log π_ref(winner)] − [log π(loser) − log π_ref(loser)]
loss   = −log sigmoid(β × margin)
```

Read it as: "the winner should have gained more probability, relative to where we started, than the loser did."

**Run it** on the SFT model from Chapter 10. The judge here prefers the shorter of two sampled responses, standing in for a human labeller so the experiment runs unattended:

In [ ]:
def judge(a, b):
    """Prefer the shorter response. A stand-in for a human comparison."""
    return (a, b) if len(a) <= len(b) else (b, a)

for step in range(300):
    qs = random.sample(prompts, 8)
    pairs = rollout(qs)                      # two sampled responses per prompt
    for q, (a, b) in zip(qs, pairs):
        w, l = judge(a, b)
        pi_w, pi_l = seq_logp(policy, chat(q), w), seq_logp(policy, chat(q), l)
        with torch.no_grad():
            rf_w, rf_l = seq_logp(ref, chat(q), w), seq_logp(ref, chat(q), l)
        margin = (pi_w - rf_w) - (pi_l - rf_l)
        losses.append(-F.logsigmoid(BETA * margin))

**What you should see:**

**Expected output:**

```
  dpo   25  loss 0.7608  margin -0.325  pref-acc(last40)   52%  [23s]
  dpo  100  loss 0.6001  margin +2.890  pref-acc(last40)   72%  [86s]
  dpo  175  loss 0.3970  margin +7.580  pref-acc(last40)   78%  [148s]
  dpo  299  loss 0.6376  margin +1.570  pref-acc(last40)   70%  [240s]

mean response length: 193 -> 42 characters (-79%)

Q: Give three tips for staying healthy.
A: '1. Exercise regularly and stay hydrated.'

Q: What is the capital of France?
A: 'France'
```

[verified]

**It learned the preference,** with preference accuracy climbing from 52%, which is chance, to around 75%.

**And then it gamed it, catastrophically.** Asked for the capital of France, the model answers **"France"**. That is a very short response. The judge rewarded shortness and got shortness, in the most literal possible way. Asked for three tips it now gives one.

This is the same failure as section 11.6, in the domain where it is most dangerous. A verifier for maths is hard to fool because arithmetic is not negotiable. A judge for "good response" is a proxy, and optimizing hard against a proxy destroys it. This is why RLHF is run gently, with a strong KL leash and few steps, and why the little book describes preference signal as shallow and easily gamed `[little-book, ch5]`.

**A methodological note you should hold me to.** My first DPO run reported a 13% reduction in response length. That number was mostly an artifact: I had batched generation with right padding, which is wrong for decoder-only models, so the measurement was corrupted. With left padding the same run showed 2%, indistinguishable from noise, and only a stronger configuration produced the unmistakable 79%. Three different numbers from the same experiment, two of them wrong. [verified]

### 11.9 Why any of this can exceed its teachers

Karpathy reaches for AlphaGo, and it is the best argument in the lecture. `[transcript]`

DeepMind's system learned Go by playing itself, with the only signal being whether it won. In the AlphaGo paper there is a plot comparing a version trained by imitating human expert moves against a version trained by reinforcement learning. The imitation version approaches human strength and stops there, which is the ceiling of Chapter 10. The RL version goes past it and keeps going.

The famous demonstration is move 37 of game two against Lee Sedol: a move human commentators initially read as a mistake, and which turned out to be the winning idea. No human demonstrator would have played it, so no amount of imitation learning could have produced it.

**That is the whole promise of this chapter.** Imitation is bounded by the demonstrator. Practice against a reward is not.

**And the whole caveat, in the same breath:** move 37 was found in a domain with a perfect, incorruptible verifier — the rules of Go. In domains without one, what you get is not move 37. It is `kaiaeaaa`, and it is "France".

### 11.10 What the field is arguing about now

RLVR is roughly two years old and unsettled. What follows is the current state, drawn from the little book's survey `[little-book, §5.3]`, and it will date faster than anything else in this book.

**Does RL create new capabilities, or surface existing ones?** Genuinely contested. Some work shows RLVR improving pass@1 while leaving pass@8 unchanged, which suggests it is learning to *select* an answer the base model could already produce rather than learning anything new. Later work with longer training and more diverse tasks reports models exceeding their base counterparts even at large k `[little-book, §5.3]`. Anyone telling you this is settled is overselling. `[uncertain]`

**High-entropy tokens do the work.** RLVR mostly affects tokens where the model was uncertain: the forking points in a reasoning chain, words like "therefore," "perhaps," "however" `[little-book, §5.3]`. Several variants (DAPO, JustRL, Lite-PPO) push harder on exactly those tokens by raising the upper clipping ratio.

**Length normalization is disputed.** GRPO divides each trajectory's gradient by its length. Some argue this is an artifact inherited from supervised learning that biases toward short correct answers and long incorrect ones; others argue it usefully encourages longer reasoning chains `[little-book, §5.3]`.

**Reward shaping for length.** Reasoning models over-expand their thinking on trivial prompts, so people add explicit length penalties. Applying one too early or too strongly collapses exploration and accuracy; more refined schemes vary the penalty with task difficulty, so the model is forced to be terse on easy prompts and allowed to think on hard ones `[little-book, §5.3]`.

**Sequence-level objectives.** The MDP can be reframed so the whole response is a single action rather than one action per token. This is arguably more principled for RLVR, and it is unstable in the naive form because the sequence-level importance ratio's variance grows multiplicatively with length. GSPO handles it with a length-normalized ratio `[little-book, §5.3]`.

### 11.11 Why this is mostly a systems problem

"While the pseudocode of GRPO fits in fewer than 30 lines, implementing an efficient RLVR training loop is a substantial systems engineering effort" `[little-book, §5.4]`. My implementation above is about 30 lines and it is a toy; the gap is infrastructure.

An RLVR system is two fleets of machines with opposite characteristics `[little-book, §5.4]`:

- The **trainer** consumes rollouts and updates weights. Compute-bound. Megatron or TorchTitan.
- The **inference engine** generates rollouts by running the policy. Memory-bandwidth-bound. vLLM or SGLang. Typically allocated several times more GPUs than the trainer, around 3:1.

They must exchange weights and rollouts continuously without either side idling. Three optimizations matter, and each trades algorithmic purity for hardware utilization:

- **Asynchrony**: let the trainer work while rollouts are still being generated.
- **Continuous batching**: refill a rollout slot the moment one finishes, rather than waiting for the slowest in the batch.
- **In-flight weight updates**: have the inference engine pull new weights as soon as they exist, so a single rollout may be generated partly by one policy and partly by the next.

Every one of these makes the data **off-policy**: collected under a policy that is no longer the one being updated, which is exactly what the PPO ratio and clipping exist to tolerate. And the problem compounds, because as training progresses the model produces longer reasoning chains, making inference disproportionately more expensive thanks to attention's quadratic cost in sequence length `[little-book, §5.4]`.

There is a subtler issue that connects back to Chapter 9's precision work: the training and inference engines are different pieces of software and do not produce bit-identical results. With mixture-of-experts models a tiny numerical difference can route a token to a different expert, so the rollout policy and the training policy diverge despite sharing weights, biasing the gradient. One fix is a second importance-sampling ratio that corrects for the mismatch explicitly `[little-book, §5.4]`.

> **For the PhD in the room.** A few things worth being precise about. GRPO's group-mean baseline is unbiased in the same way any action-independent baseline is, but dividing by the group standard deviation is not innocent: it rescales the effective step size per prompt by the inverse difficulty spread, which is part of why Dr. GRPO removes it. The KL term as implemented above is the k1 estimator (log π − log π_ref), which is unbiased for the KL but high variance and can go negative on a sample; the k3 estimator is the usual production choice. The clipping in PPO/GRPO is not a trust region in the TRPO sense — it provides no monotonic improvement guarantee, it is a heuristic surrogate that happens to work, and the performance-difference lemma it approximates is only exact when the state distributions of the two policies coincide, which is why the sequence-level formulation is cleaner: there the initial state distribution depends only on the prompt dataset, so that approximation becomes exact `[little-book, §5.3]`. Finally, the credit-assignment structure here is degenerate in an interesting way: with a terminal-only reward and deterministic transitions, every token in a trajectory receives the same advantage, so GRPO assigns credit uniformly across a response and relies on averaging over many rollouts to sort out which tokens actually mattered.

### Exercises

1. **Change the verifier** to something else, names ending in `-son`, names of exactly six letters, and confirm GRPO finds it. This is the point: the algorithm never knew what the old rule was.
2. **Make the task impossible at first.** Require names starting with `xq`. Watch the reward stay at zero forever and explain why, using the step-0 argument in 11.5.
3. **Sweep β** across 0, 0.02, 0.2, 2.0, 20.0 and plot reward against naturalness. Find where the leash starts to bind.
4. **Break the baseline.** Replace `A = R - R.mean()` with `A = R` and observe the training destabilize. That one line is the variance-reduction trick from 11.4.
5. **Shrink the group.** Try G = 4 instead of 64. The mean of four samples is a noisy baseline; see how much slower learning becomes.
6. **Fix the DPO judge.** Replace "prefer shorter" with something that rewards following the instruction, for example checking that a request for three items produces three items, and see whether "France" stops happening.

### Troubleshooting

| Symptom | Cause |
|---|---|
| Reward stays exactly 0 | No rollout ever succeeds, so every advantage is 0. Make the task easier or the group larger |
| Reward stuck at the starting rate | All rewards identical within each group; same problem, no signal |
| Output becomes gibberish | Reward hacking. Raise β, lower the learning rate, stop earlier |
| Reward climbs then collapses | Learning rate too high; the clipped ratio cannot save you from a bad optimizer setting |
| KL term appears to do nothing | It is too small relative to the policy-gradient term. Print both before tuning β |
| Loss goes negative | Expected for the policy-gradient term; it is not a likelihood and its sign means nothing on its own |
| Generation and training disagree | Different padding, sampling settings, or precision between the two paths |

### 30-second version

Fine-tuning copies demonstrations and is capped by whoever wrote them. Reinforcement learning replaces the demonstrations with a score: generate several attempts, keep what scored above the group average, and repeat. That is GRPO, and 30 lines of it took a model from satisfying a rule 0.6% of the time to 96%, with zero examples of the rule. Where the score can be checked mechanically, as in maths and code, this is how reasoning models are made. Where it cannot, you fit a judge to human preferences and the model games it: mine learned that short answers were preferred and started replying "France" when asked for the capital of France.

---